# SPLADE (2021)
---
[[paper]](https://arxiv.org/abs/2109.10086)<br>
SPLADE = Sparse Lexical and Expansion Model for Information Retrieval

# Описание
__SPLADE__ — это метод Learned Sparse Retrieval, который превращает текст в разреженные векторы высокой размерности (равной размеру словаря токенизатора). Модель одновременно решает задачи взвешивания термов (Term Weighting) и расширения документа (Document Expansion), позволяя использовать преимущества семантического поиска в рамках классического инвертированного индекса.

# Задача
Решается задача First Stage Retrieval: эффективный поиск топ-K релевантных документов в коллекции из миллионов записей по текстовому запросу.

# Мотивация
Классические методы типа BM25 (1994) страдают от проблемы Lexical Gap — если в запросе "автомобиль", а в документе "машина", BM25 их не сопоставит. Dense Retrieval (например, DPR, 2020) решает это через эмбеддинги, но требует огромных ресурсов для хранения векторных индексов и сложен в интерпретации. Существовавшие ранее методы расширения (Doc2query, 2019) работали на этапе предобработки и часто вносили шум, генерируя лишние слова, а методы взвешивания (DeepCT, 2019) не умели добавлять новые слова, которых нет в исходном тексте.

# Альтернативы
- BM25: использует только статистику частот слов, игнорируя семантику.
- Doc2query (2019): T5-модель генерирует гипотетические вопросы к документу и дописывает их в конец. Минус: отдельная тяжелая стадия генерации, сложно контролировать количество добавляемых токенов.
- DeepCT (2019): BERT предсказывает веса только для существующих в тексте слов. Не решает проблему Lexical Gap.
- DPR (2020): Dense Retrieval подход. Требует Approximate Nearest Neighbor (ANN) поиска, потребляет много RAM (десятки ГБ для MS MARCO).

# Идея
Авторы предложили использовать Masked Language Modeling (MLM) голову предобученного Трансформера для того, чтобы проецировать токены документа в пространство всего словаря (Vocabulary Space). Вместо того чтобы предсказывать пропущенное слово, модель предсказывает "важность" каждого слова из словаря для данного текста. Если слово семантически близко к тексту, его вес будет отличным от нуля, даже если самого слова в тексте нет.

# Архитектура
Модель строится на базе BERT.
1.  Encoder: BERT обрабатывает последовательность токенов $t \in D$.
2.  Logits: Для каждого токена на выходе BERT берется вектор размерности словаря $V$ (через стандартную MLM голову).
3.  Transformation: К логитам применяется $log(1 + ReLU(w))$, чтобы оставить только положительные значения и "прижать" слишком большие веса.
4.  Pooling: Выполняется Max Pooling по всем токенам документа. Для каждого слова $j$ из словаря $V$ итоговый вес $w_j = \max_{i \in D} w_{i,j}$.
5.  Sparsity: На выходной вектор накладывается регуляризация, чтобы большинство весов стали нулевыми.

# Обучение
Процесс обучения объединяет две функции потерь:
1.  Ranking Loss: Обычно используется InfoNCE или Contrastive Loss. Модель учится максимизировать скалярное произведение векторов запроса и релевантного документа по сравнению с нерелевантными (In-batch Negatives или Hard Negatives).
2.  Sparsity Loss: Применяется регуляризация FLOPs (Paria et al., 2020) или L1. Это критически важный компонент: без него вектор будет плотным (Dense). FLOPs loss минимизирует ожидаемое количество операций при поиске, заставляя модель обнулять веса маловажных токенов.

# Инференс
1.  Indexing (Offline): Каждый документ коллекции пропускается через SPLADE. На выходе получается разреженный вектор (например, из 30 000 измерений только 100-200 имеют ненулевые значения). Эти значения сохраняются в обычный инвертированный индекс (ElasticSearch/Lucene) как веса термов.
2.  Query Encoding (Online): Запрос пропускается через ту же модель SPLADE, превращаясь в такой же разреженный вектор.
3.  Retrieval: Выполняется стандартный поиск в инвертированном индексе. Оценка релевантности — это сумма весов совпавших токенов (Dot Product).

# Результаты
Сравнение на датасете MS MARCO Passage Ranking:
- Эффективность: SPLADE v2 достигла MRR@10 = 0.369. Для сравнения: BM25 = 0.187, а мощный Dense-энкодер DPR = 0.320.
- Хранение: Индекс SPLADE занимает примерно в 4-10 раз меньше места, чем плотные векторы DPR (за счет разреженности и хранения только ID токенов и их весов).
- Интерпретируемость: В отличие от Dense моделей, здесь можно посмотреть, какие именно слова "активировала" модель. Например, для текста про "коронавирус" модель может сама добавить веса словам "пандемия", "вирус", "маска", что наглядно объясняет результат поиска.

## 📝 Критический анализ

```markdown
# SPLADE (2021)
---
[[paper]](https://arxiv.org/abs/2109.10086)<br>
SPLADE = Sparse Lexical and Expansion Model for Information Retrieval

## Описание
**SPLADE** — метод Learned Sparse Retrieval, преобразующий текст в разреженные векторы высокой размерности. Модель решает задачи взвешивания термов и расширения документа, используя преимущества семантического поиска в рамках классического инвертированного индекса.

## Задача
Решается задача First Stage Retrieval: эффективный поиск топ-K релевантных документов в коллекции из миллионов записей по текстовому запросу.

## Мотивация
Классические методы, такие как BM25 (1994), страдают от проблемы Lexical Gap. Dense Retrieval, например, DPR (2020), требует значительных ресурсов для хранения и сложен в интерпретации. Методы расширения, такие как Doc2query (2019), часто вносят шум, а методы взвешивания, такие как DeepCT (2019), не решают проблему Lexical Gap.

## Альтернативы
- BM25: игнорирует семантику.
- Doc2query (2019): требует отдельной стадии генерации.
- DeepCT (2019): не решает Lexical Gap.
- DPR (2020): требует много RAM для ANN поиска.

## Идея
Использование Masked Language Modeling (MLM) головы предобученного Трансформера для проекции токенов документа в пространство всего словаря. Модель предсказывает "важность" каждого слова из словаря для текста.

## Архитектура
- **Encoder**: BERT обрабатывает токены $t \in D$.
- **Logits**: Вектор размерности словаря $V$ через MLM голову.
- **Transformation**: Применяется $log(1 + ReLU(w))$.
- **Pooling**: Max Pooling по токенам документа.
- **Sparsity**: Регуляризация для обнуления большинства весов.

<img src="img/img.png" width=500>

## Обучение
Объединяет две функции потерь:
1. **Ranking Loss**: Максимизация скалярного произведения векторов запроса и релевантного документа.
2. **Sparsity Loss**: Регуляризация FLOPs или L1 для минимизации операций при поиске.

## Инференс
- **Indexing (Offline)**: Документы проходят через SPLADE, создавая разреженные векторы, сохраняемые в инвертированный индекс.
- **Query Encoding (Online)**: Запрос превращается в разреженный вектор.
- **Retrieval**: Стандартный поиск в инвертированном индексе, оценка релевантности — сумма весов совпавших токенов.

## Результаты
На MS MARCO Passage Ranking:
- **Эффективность**: SPLADE v2 достигла MRR@10 = 0.369, превосходя BM25 и DPR.
- **Хранение**: Индекс SPLADE занимает в 4-10 раз меньше места, чем DPR.
- **Интерпретируемость**: Можно увидеть, какие слова "активировала" модель, например, для текста про "коронавирус" добавляются веса словам "пандемия", "вирус", "маска".
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
import torch.nn.functional as F

# SPLADE: Sparse Lexical and Expansion Model for Information Retrieval
# This example illustrates the key concepts of SPLADE using a simplified version.

# Initialize BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Sample document
document = "The car is fast and efficient."

# Tokenize the document
inputs = tokenizer(document, return_tensors='pt')

# Pass the tokens through BERT
outputs = model(**inputs)

# Extract the last hidden state
last_hidden_state = outputs.last_hidden_state

# SPLADE Transformation: Apply the MLM head to project tokens into the vocabulary space
# For simplicity, we use the hidden state directly as logits
logits = last_hidden_state

# Transformation: Apply log(1 + ReLU(w)) to logits
transformed_logits = torch.log1p(F.relu(logits))

# Pooling: Max Pooling over all tokens for each word in the vocabulary
pooled_weights = torch.max(transformed_logits, dim=1).values

# Sparsity: Apply L1 regularization to encourage sparsity
# In practice, this would be part of the loss function during training
sparsity_loss = torch.norm(pooled_weights, p=1)

# Display the resulting sparse vector
# Note: In a real implementation, this vector would be used to create an inverted index
print("Sparse Vector Representation (Top 10 non-zero weights):")
non_zero_indices = torch.nonzero(pooled_weights, as_tuple=True)[1]
non_zero_weights = pooled_weights[0, non_zero_indices]
top_indices = non_zero_weights.topk(10).indices
for idx in top_indices:
    token_id = non_zero_indices[idx].item()
    token_weight = non_zero_weights[idx].item()
    token = tokenizer.convert_ids_to_tokens(token_id)
    print(f"Token: {token}, Weight: {token_weight:.4f}")

# Inference: Query Encoding
query = "fast car"
query_inputs = tokenizer(query, return_tensors='pt')
query_outputs = model(**query_inputs)
query_logits = query_outputs.last_hidden_state
query_transformed_logits = torch.log1p(F.relu(query_logits))
query_pooled_weights = torch.max(query_transformed_logits, dim=1).values

# Retrieval: Compute dot product between query and document sparse vectors
# This is a simplified version of the retrieval process
retrieval_score = torch.dot(pooled_weights.flatten(), query_pooled_weights.flatten())
print(f"Retrieval Score: {retrieval_score.item():.4f}")

# Note: In a full implementation, the sparse vectors would be stored in an inverted index,
# and retrieval would involve searching this index for top-K relevant documents.
```

### Key Concepts Illustrated:
1. **Sparse Vector Representation**: The document is transformed into a sparse vector using BERT's MLM head, where each dimension corresponds to a vocabulary token.
2. **Transformation and Pooling**: Logarithmic transformation and max pooling are applied to logits to obtain the final sparse vector.
3. **Sparsity**: L1 regularization is used to encourage sparsity in the vector, making it suitable for inverted index storage.
4. **Query Encoding and Retrieval**: The same process is applied to the query, and a dot product is used to compute the retrieval score, simulating the retrieval process in an inverted index.

This example provides a simplified view of SPLADE's core mechanisms, focusing on the transformation of text into sparse vectors and the retrieval process.